# Image Captioning with CNN + LSTM
**Course:** Machine Learning — Homework 2  
**Dataset:** Flickr8k | **Encoder:** ResNet50 | **Decoder:** LSTM + GloVe


## A. Data Collection
Download and load the **Flickr8k** dataset (8,000 images, each with 5 captions).  
Captions are parsed from `captions.txt` and stored in a dictionary keyed by image filename.  
A few samples are displayed to get an overall intuition of the data.


In [34]:
import os, json
import pandas as pd
from collections import Counter
import pickle

BASE_PATH = "."

def load_captions(filepath):
    df = pd.read_csv(filepath)
    captions = {}
    for _, row in df.iterrows():
        img_id = row['image'].split('.')[0]
        captions.setdefault(img_id, []).append(row['caption'])
    return captions

captions_dict = load_captions(os.path.join(BASE_PATH, "captions.txt"))
print(f"pictures count: {len(captions_dict)}")



pictures count: 8091


## B. Caption Preprocessing
Raw captions are cleaned with the following steps (using Python's `string` library):
- Convert to **lowercase**
- Remove **punctuation**
- Remove **tokens containing digits**
- Remove **single-character words**

Cleaned captions are saved to disk to avoid recomputation and reduce cache usage.


In [16]:
import string

def clean_caption(caption):
    caption = caption.lower().translate(str.maketrans('', '', string.punctuation))
    return ' '.join(w for w in caption.split() if w.isalpha() and len(w) > 1)

cleaned_captions = {img_id: [clean_caption(c) for c in caps]
                    for img_id, caps in captions_dict.items()}

with open("cleaned_captions.pkl", "wb") as f:
    pickle.dump(cleaned_captions, f)



## C. Vocabulary Building
A vocabulary is built from all unique words across all cleaned captions.  
Words appearing **fewer than 10 times** are discarded (low-frequency words add noise and inflate the vocab size).  
The final vocabulary size is printed.


In [17]:
with open("cleaned_captions.pkl", "rb") as f:
    cleaned_captions = pickle.load(f)

word_freq = Counter(w for caps in cleaned_captions.values()for cap in caps for w in cap.split())
vocab = {w for w, c in word_freq.items() if c >= 10}
print(f"vocabulary size: {len(vocab)}")

vocabulary size: 1947


## D. Train Split & START/END Tokens
The file `Flickr_8k.trainImages.txt` lists the 6,000 training image filenames.  
`<START>` and `<END>` tokens are prepended/appended to each caption — these are required by the LSTM decoder  
to know when to begin and stop generating text.  
Processed train captions are saved to disk.


In [19]:
import random

df = pd.read_csv(os.path.join(BASE_PATH, "captions.txt"))
all_ids = [x.split('.')[0] for x in df['image'].unique()]
random.seed(42)
random.shuffle(all_ids)

train_ids = all_ids[:6000]
test_ids  = all_ids[6000:]

with open("Flickr_8k.trainImages.txt", "w") as f:
    f.write("\n".join(train_ids))
with open("Flickr_8k.testImages.txt", "w") as f:
    f.write("\n".join(test_ids))

print(f"Train: {len(train_ids)}, Test: {len(test_ids)}")

train_img_ids = set(train_ids)
train_captions = {img_id: [f"<START> {cap} <END>" for cap in caps]
                  for img_id, caps in cleaned_captions.items()
                  if img_id in train_img_ids}

print(f"train pictures count: {len(train_captions)}")


Train: 6000, Test: 2091
train pictures count: 6000


## E. Image Feature Extraction (Transfer Learning)
A **pre-trained ResNet50** (trained on ImageNet) is used as the image encoder.  
The final classification layer is removed so the model outputs a **2048-d feature vector** per image.  
This is the `encode()` function. Features for all train and test images are extracted once and saved to disk,  
avoiding redundant forward passes during training.


In [21]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
import numpy as np
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

base_model = models.resnet50(pretrained=True)
encoder_model = torch.nn.Sequential(*list(base_model.children())[:-1])
encoder_model.eval().to(device)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def encode_images(img_ids, img_dir, save_path):
    features = {}
    with torch.no_grad():
        for img_id in img_ids:
            img_path = os.path.join(img_dir, img_id + ".jpg")  # ← اصلاح شد
            if not os.path.exists(img_path):
                continue
            img = Image.open(img_path).convert("RGB")
            feat = encoder_model(transform(img).unsqueeze(0).to(device))
            features[img_id] = feat.squeeze().cpu().numpy()
    with open(save_path, "wb") as f:
        pickle.dump(features, f)
    print(f"encoded {len(features)}/{len(img_ids)} images → {save_path}")
    return features

img_dir = os.path.join(BASE_PATH, "Images")

# بررسی تطابق نام فایل‌ها با img_id

sample_files = os.listdir(img_dir)[:3]
print("sample files in Images in d dimention:", sample_files)
print("sample train_ids:", train_ids[:3])

train_features = encode_images(train_ids, img_dir, "train_features.pkl")
test_features  = encode_images(test_ids,  img_dir, "test_features.pkl")

if train_features:
    print(f"train features: {len(train_features)}, shape: {next(iter(train_features.values())).shape}")
else:
    print("train_features خالیه! نام فایل‌ها رو بررسی کن.")


sample files in Images in d dimention: ['2387197355_237f6f41ee.jpg', '2609847254_0ec40c1cce.jpg', '2046222127_a6f300e202.jpg']
sample train_ids: ['2874984466_1aafec2c9f', '519228867_2fd25e38d4', '3163198309_bbfe504f0a']
encoded 6000/6000 images → train_features.pkl
encoded 2091/2091 images → test_features.pkl
train features: 6000, shape: (2048,)


## F. Word Dictionaries (word2idx / idx2word)
Two dictionaries are built manually (no pre-built tokenizer):
- `word2idx`: maps each vocabulary word to a unique integer ID
- `idx2word`: reverse mapping

Special tokens `<PAD>`, `<START>`, `<END>`, `<UNK>` are added.  
The **maximum caption length** across all training captions is also computed — this is needed for padding in the next step.


In [22]:
special_tokens = ["<PAD>", "<START>", "<END>", "<UNK>"]
full_vocab = special_tokens + sorted(list(vocab))

word2idx = {w: i for i, w in enumerate(full_vocab)}
idx2word = {i: w for w, i in word2idx.items()}

VOCAB_SIZE = len(word2idx)
print(f"vocab final size: {VOCAB_SIZE}")

# بیشترین طول caption
max_len = max(
    len(cap.split())
    for caps in train_captions.values()
    for cap in caps
)
print(f"caption max lenght: {max_len}")


vocab final size: 1951
caption max lenght: 34


## G. Data Generator (Mini-batch Iterator)
Captions are **padded** with `<PAD>` to a fixed length using `pad_sequence`.  
The generator (`DataGenerator`) works as an iterator and yields mini-batches of:
- **Input 1:** image feature vector
- **Input 2:** partial caption sequence (all words before the target word)
- **Output (target):** next word in the sequence

This teacher-forcing setup is the standard approach for training sequence-to-sequence models.


In [23]:
import torch
from torch.utils.data import Dataset, DataLoader

def pad_sequence_manual(seq, max_len, pad_idx=0):
    return seq + [pad_idx] * (max_len - len(seq))

class CaptionDataset(Dataset):
    def __init__(self, captions, features, word2idx, max_len):
        self.data = []
        for img_id, caps in captions.items():
            if img_id not in features:
                continue
            feat = features[img_id]
            for cap in caps:
                tokens = [word2idx.get(w, word2idx["<UNK>"]) for w in cap.split()]
                # برای هر کلمه هدف: ورودی = تمام قبلی‌ها + تصویر
                for i in range(1, len(tokens)):
                    in_seq = pad_sequence_manual(tokens[:i], max_len, word2idx["<PAD>"])
                    out_word = tokens[i]
                    self.data.append((feat, in_seq, out_word))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        feat, seq, target = self.data[idx]
        return (
            torch.tensor(feat, dtype=torch.float32),
            torch.tensor(seq, dtype=torch.long),
            torch.tensor(target, dtype=torch.long)
        )

with open("train_features.pkl", "rb") as f:
    train_features = pickle.load(f)

dataset = CaptionDataset(train_captions, train_features, word2idx, max_len)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)
print(f"Data Generator sample count: {len(dataset)}")


Data Generator sample count: 305924


## H. GloVe Embeddings (Pre-trained Word Vectors)
**GloVe** (200-d) pre-trained word vectors are loaded to initialize the embedding layer.  
An `embedding_matrix` of shape `(vocab_size, 200)` is built by looking up each vocabulary word in GloVe.  
Words not found in GloVe get a random vector.  

The embedding layer is then **frozen** (`requires_grad=False`) so GloVe weights are not updated during training —  
this preserves the semantic structure of the pre-trained space.


In [24]:
EMBED_DIM = 200  # glove.6B.200d

def load_glove(glove_path, embed_dim):
    glove = {}
    with open(glove_path, 'r', encoding='utf-8') as f:
        for line in f:
            vals = line.split()
            glove[vals[0]] = np.array(vals[1:], dtype=np.float32)
    return glove

glove_vectors = load_glove("glove.6B.200d.txt", EMBED_DIM)

embedding_matrix = np.zeros((VOCAB_SIZE, EMBED_DIM))
for word, idx in word2idx.items():
    if word in glove_vectors:
        embedding_matrix[idx] = glove_vectors[word]

embedding_matrix = torch.tensor(embedding_matrix, dtype=torch.float32)
print(f"embedding matrix shape: {embedding_matrix.shape}")


embedding matrix shape: torch.Size([1951, 200])


## I. Model Architecture (CNN Encoder + LSTM Decoder)
The model combines:
- **Image encoder:** a linear projection of the ResNet50 feature (2048-d → embed_dim)
- **Word embedding:** frozen GloVe layer
- **LSTM decoder:** takes the concatenation of image feature and word embeddings as input
- **Output layer:** linear projection to vocabulary size, followed by softmax during inference

At each time step, the LSTM receives the previous word embedding and uses the image feature  
as the initial hidden/cell state (or concatenated input), then predicts the next word.


In [25]:
import torch.nn as nn

class ImageCaptionModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, feat_dim=2048):
        super().__init__()
        self.image_fc = nn.Linear(feat_dim, hidden_dim)
        
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.embedding.weight = nn.Parameter(embedding_matrix)
        self.embedding.weight.requires_grad = False  # freeze GloVe
        
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim * 2, vocab_size)
        self.dropout = nn.Dropout(0.3)

    def forward(self, img_feat, seq):
        img_emb = torch.relu(self.image_fc(img_feat))          # (B, hidden)
        word_emb = self.dropout(self.embedding(seq))           # (B, T, embed)
        lstm_out, _ = self.lstm(word_emb)                      # (B, T, hidden)
        # الحاق ویژگی تصویر به هر گام زمانی
        img_expanded = img_emb.unsqueeze(1).expand(-1, lstm_out.size(1), -1)
        combined = torch.cat([lstm_out, img_expanded], dim=-1) # (B, T, hidden*2)
        out = self.fc_out(combined[:, -1, :])                  # فقط آخرین گام
        return out


HIDDEN_DIM = 256
model = ImageCaptionModel(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM).to(device)
print(model)


ImageCaptionModel(
  (image_fc): Linear(in_features=2048, out_features=256, bias=True)
  (embedding): Embedding(1951, 200)
  (lstm): LSTM(200, 256, batch_first=True)
  (fc_out): Linear(in_features=512, out_features=1951, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
)


## J. Training Loop with TensorBoard Logging
The model is trained for **10 epochs** using:
- **Optimizer:** Adam
- **Loss:** CrossEntropyLoss — `<PAD>` tokens are ignored (`ignore_index`) so they don't contribute to the gradient

At each epoch, **loss** and **token-level accuracy** are logged to TensorBoard (`runs/` directory).  
The TensorBoard summary files must be included in the final submission.


In [26]:
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=word2idx["<PAD>"])

from torch.utils.tensorboard import SummaryWriter
writer = SummaryWriter("runs/captioning")

EPOCHS = 10
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for img_feat, seq, target in dataloader:
        img_feat, seq, target = img_feat.to(device), seq.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(img_feat, seq)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (output.argmax(dim=-1) == target).sum().item()
        total += target.size(0)
    avg_loss = total_loss / len(dataloader)
    accuracy = correct / total
    writer.add_scalar("Loss/train", avg_loss, epoch)
    writer.add_scalar("Accuracy/train", accuracy, epoch)
    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {avg_loss:.4f} - Acc: {accuracy:.4f}")

model.eval()
correct, total = 0, 0
with torch.no_grad():
    for img_feat, seq, target in dataloader:
        img_feat, seq, target = img_feat.to(device), seq.to(device), target.to(device)
        output = model(img_feat, seq)
        preds = output.argmax(dim=-1)
        correct += (preds == target).sum().item()
        total += target.size(0)
acc = correct / total
writer.add_scalar("Accuracy/train", acc, epoch)
print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {avg_loss:.4f} - Acc: {acc:.4f}")

writer.close()
torch.save(model.state_dict(), "caption_model.pth")

Epoch 1/10 - Loss: 4.2636 - Acc: 0.2034
Epoch 2/10 - Loss: 3.4344 - Acc: 0.2841
Epoch 3/10 - Loss: 3.2168 - Acc: 0.3063
Epoch 4/10 - Loss: 3.0911 - Acc: 0.3189
Epoch 5/10 - Loss: 3.0074 - Acc: 0.3280
Epoch 6/10 - Loss: 2.9413 - Acc: 0.3363
Epoch 7/10 - Loss: 2.8922 - Acc: 0.3416
Epoch 8/10 - Loss: 2.8477 - Acc: 0.3463
Epoch 9/10 - Loss: 2.8136 - Acc: 0.3498
Epoch 10/10 - Loss: 2.7758 - Acc: 0.3543
Epoch 10/10 - Loss: 2.7758 - Acc: 0.3784


## K. Inference — Generating Captions on Test Images
The `generate_caption` function uses **greedy decoding**:  
starting from `<START>`, the model predicts the most probable next word at each step,  
feeds it back as input, and stops when `<END>` is generated or the maximum length is reached.

Several test images are captioned and displayed alongside the generated text for qualitative evaluation.


In [33]:
def generate_caption(model, img_feat, word2idx, idx2word, max_len):
    model.eval()
    seq = [word2idx["<START>"]]
    img_tensor = torch.tensor(img_feat, dtype=torch.float32).unsqueeze(0).to(device)
    
    with torch.no_grad():
        for _ in range(max_len):
            seq_padded = pad_sequence_manual(seq, max_len, word2idx["<PAD>"])
            seq_tensor = torch.tensor([seq_padded], dtype=torch.long).to(device)
            output = model(img_tensor, seq_tensor)
            next_word_idx = output.argmax(dim=-1).item()
            seq.append(next_word_idx)
            if idx2word[next_word_idx] == "<END>":
                break
    
    words = [idx2word[i] for i in seq[1:] if idx2word[i] not in ["<END>", "<PAD>"]]
    return ' '.join(words)

# اعمال روی چند تصویر تست
with open("test_features.pkl", "rb") as f:
    test_features = pickle.load(f)

model.load_state_dict(torch.load("caption_model.pth"))

for img_id, feat in list(test_features.items())[:10]:
    caption = generate_caption(model, feat, word2idx, idx2word, max_len)
    print(f"{img_id}: {caption}")

2281054343_95d6d3b882: man in <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK>
2355819665_39021ff642: young boy in blue shirt is riding bike in the grass
2502856739_490db7a657: dog runs through the grass
2198511848_311d8a8c2f: man is jumping off cliff into the air
494221578_027f51cdf4: little girl in blue and yellow shirt sliding down slide
2450486758_a66fd296ea: group of people are playing in the street
2201978994_c444e64810: two children are playing in the snow
522486784_978021d537: two dogs are playing in the grass
3500399969_f54ce5848f: two children are jumping into the air in front of pond
480607352_65614ab348: two people are walking through the street


2281054343_95d6d3b882: man in hat and hat is standing on busy street

2355819665_39021ff642: man in blue shirt and helmet is riding bike

2502856739_490db7a657: black dog is running through the grass

2198511848_311d8a8c2f: snowboarder in midair

494221578_027f51cdf4: little boy slides down slide